In [1]:
from ccka.models.kernel import KernelModel
from ccka.circuits.angleEmbeddingKernel import quackEmbeddingCircuit
from ccka.aligner.kta import fullKTA
import pennylane as qml
import jax
import jax.numpy as jnp

In [2]:
data = jnp.load('../data/checkerboard_dataset.npy', allow_pickle=True).item()
X = jnp.asarray(data['x_train'])
y = jnp.asarray(data['y_train'])
N, D = X.shape
x1 = jnp.repeat(X, N, axis=0)      # (N*N, D)
x2 = jnp.tile(X, (N, 1))

In [3]:
x1.shape, x2.shape

((900, 2), (900, 2))

In [4]:
kernel = quackEmbeddingCircuit(
                                num_qubits = 5,
                                reps = 6,
                                reupload = True
                        )
init_weights = kernel.init_weights()
model = KernelModel(circuit = kernel)

In [5]:
aligner = fullKTA(
                    kernel_model= model,
                    data = X,
                    labels = y,
                    matrix_type='regular',
                    split_size=0.5,
                    landmark_points=10,
                    learning_rate=0.01,
                    optimizer= 'adam',
                    epochs=500
)

In [6]:
history = aligner.align()

Aligning Kernel with Full Kernel KTA: 100%|██████████| 500/500 [00:30<00:00, 16.19it/s]

┌────────────────────────────────────────────────────────────────────────────┐
│              FULL KERNEL TARGET ALIGNMENT – TRAINING SUMMARY               │
├────────────────────────────────────────────────────────────────────────────┤
│ 'Epochs run           : 501'                                               │
│ 'Total training time  : 30.90 seconds'                                     │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│                              ACCURACY METRICS                              │
├────────────────────────────────────────────────────────────────────────────┤
│ 'Initial train accuracy : 0.7333'                                          │
│ 'Final train accuracy   : 1.0000'                                          │
│ 'Best train accuracy    : 1.0000'                                          │
│ 'Initial test accuracy  : 0.4000'                